In [76]:
import pandas as pd
import numpy as np


print('hello world')

df = pd.read_csv('data/raw/Employee.csv')

display(df.sample(10))


hello world


,employee_id,age,salary,department,tenure_years,performance_score,left_company
253,EMP0257,53.2,47685.91,Engineering,1.0,4.0,0
226,EMP0279,NaN,32821.21,Finance,2.7,2.0,1
171,EMP0132,NaN,24411.52,Sales,2.4,3.0,0
214,EMP0136,56.6,56111.42,Engineering,2.4,4.0,0
153,EMP0084,31.8,47113.67,Engineering,0.1,4.0,0
21,EMP0270,39.6,49727.30,HR,2.2,4.0,0
158,EMP0134,43.7,65763.13,Sales,23.4,4.0,0
123,EMP0171,NaN,50874.22,Engineering,0.6,4.0,0
80,EMP0165,49.6,53315.08,Engineering,8.1,4.0,0
177,EMP0042,40.1,58655.31,Sales,8.4,2.0,0


In [77]:
print(f"Rows: {df.shape[0]}  Columns: {df.shape[1]}")
display(df.info())

Rows: 303  Columns: 7
<class 'pandas.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   employee_id        303 non-null    str    
 1   age                279 non-null    float64
 2   salary             276 non-null    float64
 3   department         303 non-null    str    
 4   tenure_years       303 non-null    float64
 5   performance_score  280 non-null    float64
 6   left_company       303 non-null    int64  
dtypes: float64(4), int64(1), str(2)
memory usage: 16.7 KB


None

In [78]:
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(1)
null_summary = pd.DataFrame({'nulls': null_counts, 'pct': null_pct})
display(null_summary[null_summary['nulls'] > 0])
display("Null values in each column:", null_counts) 

,nulls,pct
age,24,7.9
salary,27,8.9
performance_score,23,7.6


'Null values in each column:'

employee_id           0
age                  24
salary               27
department            0
tenure_years          0
performance_score    23
left_company          0
dtype: int64

In [79]:
display(df.describe().round(2))

,age,salary,tenure_years,performance_score,left_company
count,279.00,276.00,303.00,280.00,303.00
mean,41.75,91819.19,4.29,3.40,0.15
std,59.45,599227.35,4.24,0.98,0.36
min,-4.00,-500.00,-2.50,1.00,0.00
25%,29.75,44110.68,1.00,3.00,0.00
50%,38.50,55133.78,3.10,3.00,0.00
75%,45.80,64169.79,6.05,4.00,0.00
max,999.00,9999999.00,23.40,5.00,1.00


check if column AGE is missing accross departments or does it have a relations

In [80]:
df['age_missing'] = df['age'].isnull().astype(int)
display(df.groupby('department')['age_missing'].mean().round(2))


department
Engineering    0.04
FINANCE        0.00
Finance        0.10
HR             0.08
Hr             0.00
Marketing      0.12
SALES          0.00
Sales          0.09
engineering    0.14
marketing      0.00
Name: age_missing, dtype: float64

from what i can see the null values are accross the whole department and its just noises. can be filled up or just use imputer on a pipeline for model.

# Check if there is a relation in missing values of salary to department col

In [81]:
df['missing_salary'] = df['salary'].isnull().astype(int)
display(df.groupby('department')['missing_salary'].mean().round(2))

department
Engineering    0.00
FINANCE        0.00
Finance        0.26
HR             0.36
Hr             1.00
Marketing      0.00
SALES          0.00
Sales          0.00
engineering    0.00
marketing      0.00
Name: missing_salary, dtype: float64

for what i can see i can only found 2 departments that has missing values so i feel like there might be a relation on this one
- it is possible that they are not noises and might be a structure on why they are missing
- i believe this needs to be checked and expand research before proceeding to the model

# CHECK IF THE NULL VALUES FROM PERFORMANCE_SCORE HAS RELATIONS TO LEFT COMPANY

In [82]:
df['missing_performance_score'] = df['performance_score'].isnull().astype(int)
display(df.groupby('left_company')['missing_performance_score'].mean().round(2))

left_company
0    0.00
1    0.51
Name: missing_performance_score, dtype: float64

This one is rather obvious for the structure, it is best to assume that the missingness on the performance score is because they left the company. assuming all of the missingness has connections.

# use iqr for numerical testing of outliers

In [83]:
def iqr_bounds(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return lower, upper

# display(df.columns)
for col in ['age', 'salary', 'tenure_years']:
    low, up = iqr_bounds(df[col])
    outliers = df[(df[col] < low) | (df[col] > up)]
    print(f"\n{col}: {low:.2f} - {up:.2f} are normal values")
    print(f"{col}: {len(outliers)} outliers")
    if len(outliers):
        print(outliers[[col, 'department']].sort_values(col))


age: 5.68 - 69.88 are normal values
age: 6 outliers
       age   department
71    -4.0           HR
149   -0.9  Engineering
0     70.6        Sales
208   84.2        Sales
10   187.0        Sales
267  999.0        Sales

salary: 14022.02 - 94258.45 are normal values
salary: 4 outliers
         salary   department
147     -500.00  engineering
109   101183.21        Sales
94    450000.00           HR
126  9999999.00        Sales

tenure_years: -6.57 - 13.62 are normal values
tenure_years: 15 outliers
     tenure_years   department
209          13.8  Engineering
23           13.9  Engineering
295          13.9    Marketing
237          14.2  Engineering
147          14.5  engineering
72           14.7  Engineering
154          14.9      Finance
174          15.9  Engineering
20           16.5           HR
246          17.3  Engineering
22           17.6    Marketing
166          18.9        Sales
168          21.1        Sales
288          22.4  Engineering
158          23.4        Sales

From what i can see from the iqr output: 
- age column might actually be valid since the numbers have negative values which you might never see this column moreover there are ages that far exceeds the human capabilities but still there are 2 values that needs to be checked

- salary i think is valid, negative values and the other 3 are high earners maybe. they might be the owner of company so needs to check also

- tenure_years column has some invalid output with iqr i'd say. from the output we cann se that there are negative values for the normal distribution which doesn't make sense and the ones it flags as outliers seems to be normal.



Using z - score method to test the outliers

In [84]:
from scipy import stats

for col in ['age', 'salary']:
    z = np.abs(stats.zscore(df[col].dropna()))
    n_out = (z > 3).sum()
    print(f'{col}: {n_out} outliers')

age: 1 outliers
salary: 1 outliers


Z-score only provides 1 outliers for age and salary, i'd say this would be normal since it the value from before has 999 for age and 9999999 salary and would safely assume that those are the value that is outside the mean

## CLEANING

## CLEANING — Correct order
Run these cells in order. Each step has a verification print so you can confirm it worked before moving to the next.

### Step 0 — Start fresh copy from raw df
Always work on a copy, never mutate the original. This way you can re-run from here without reloading the CSV.

In [85]:
df_clean = df.copy()

# Drop the diagnosis columns we created during profiling
diag_cols = ['age_missing', 'missing_salary', 'missing_performance_score']
df_clean.drop(columns=[c for c in diag_cols if c in df_clean.columns], inplace=True)

print(f'Starting shape: {df_clean.shape}')  # expect (303, 7)
print(f'Nulls:\n{df_clean.isnull().sum()}')

Starting shape: (303, 7)
Nulls:
employee_id           0
age                  24
salary               27
department            0
tenure_years          0
performance_score    23
left_company          0
dtype: int64


### Step 1 — Fix department casing
Must happen BEFORE any groupby on department. Otherwise HR, Hr, and HR are treated as three different groups.

In [86]:
df_clean['department'] = df_clean['department'].str.strip().str.title()

print('Unique departments after fix:')
print(df_clean['department'].unique())  # expect 5 clean values
print(f'Value counts:\n{df_clean["department"].value_counts()}')

Unique departments after fix:
<StringArray>
['Sales', 'Marketing', 'Engineering', 'Finance', 'Hr']
Length: 5, dtype: str
Value counts:
department
Engineering    100
Sales           67
Marketing       53
Hr              51
Finance         32
Name: count, dtype: int64


### Step 2 — Remove impossible outliers BEFORE imputing
Imputation computes medians from existing values. If 999 or 9999999 are still in the data, they skew the median. Remove them first.

- Age: clip to 18–70 (domain bounds for working employees)
- Salary: remove negative (impossible) and the 9999999 entry error. Keep 450000 — that's a real value.
- Tenure: clip to 0 minimum (can't have negative years at a company)

In [87]:
# Age: clip impossible values
df_clean['age'] = df_clean['age'].clip(lower=18, upper=70)
print(f'Age range after clip: {df_clean["age"].min()} - {df_clean["age"].max()}')

# Salary: remove impossible negatives and the clear data entry error
# We keep 450000 — could be a real executive
rows_before = len(df_clean)
df_clean = df_clean[df_clean['salary'] > 0]          # removes -500
df_clean = df_clean[df_clean['salary'] < 9000000]    # removes 9999999
print(f'Rows removed by salary filter: {rows_before - len(df_clean)}')  # expect 2

# Tenure: clip negatives to 0
df_clean['tenure_years'] = df_clean['tenure_years'].clip(lower=0)
print(f'Tenure min after clip: {df_clean["tenure_years"].min()}')  # expect 0.0

print(f'\nShape after outlier removal: {df_clean.shape}')  # expect ~(301, 7)

Age range after clip: 18.0 - 70.0
Rows removed by salary filter: 29
Tenure min after clip: 0.0

Shape after outlier removal: (274, 7)


### Step 3 — Impute missing values
Now that outliers are gone, medians are clean.

- **Age (MCAR)**: global median — missingness is random, no group structure
- **Salary (MAR)**: median per department — missingness is concentrated in HR/Finance, so imputing globally would pull those departments toward the wrong center
- **Performance score (MNAR)**: flag first, then fill — the flag column is itself a signal (it tells the model 'this person probably left because of poor performance')

In [88]:
# MCAR — Age: global median
age_median = df_clean['age'].median()
df_clean['age'] = df_clean['age'].fillna(age_median)
print(f'Age imputed with median: {age_median}')
print(f'Age nulls remaining: {df_clean["age"].isnull().sum()}')  # expect 0

# MAR — Salary: group median by department
dept_medians = df_clean.groupby('department')['salary'].transform('median')
df_clean['salary'] = df_clean['salary'].fillna(dept_medians)
print(f'\nSalary nulls remaining: {df_clean["salary"].isnull().sum()}')  # expect 0
print('Group medians used:')
print(df_clean.groupby('department')['salary'].median().round(0))

# MNAR — Performance score: flag THEN fill
df_clean['perf_was_missing'] = df_clean['performance_score'].isnull().astype(int)
perf_median = df_clean['performance_score'].median()
df_clean['performance_score'] = df_clean['performance_score'].fillna(perf_median)
print(f'\nPerf score imputed with median: {perf_median}')
print(f'Perf nulls remaining: {df_clean["performance_score"].isnull().sum()}')  # expect 0
print(f'perf_was_missing flag — 1s: {df_clean["perf_was_missing"].sum()}')  # expect ~21

Age imputed with median: 38.35
Age nulls remaining: 0

Salary nulls remaining: 0
Group medians used:
department
Engineering    53663.0
Finance        55359.0
Hr             62477.0
Marketing      52794.0
Sales          56748.0
Name: salary, dtype: float64

Perf score imputed with median: 3.0
Perf nulls remaining: 0
perf_was_missing flag — 1s: 21


### Step 4 — Remove duplicates
We ignore `employee_id` when checking duplicates because the IDs are unique by design — two rows with the same data but different IDs are still duplicates.

**Common mistake**: `print(len(df) - len(df_clean))` counts ALL rows removed since the start, not just from this step. Always count duplicates before dropping them.

In [89]:
# Count duplicates BEFORE dropping
n_dupes = df_clean.duplicated(subset=df_clean.columns.difference(['employee_id'])).sum()
print(f'Duplicate rows found: {n_dupes}')  # expect 2-3

df_clean = df_clean.drop_duplicates(
    subset=df_clean.columns.difference(['employee_id'])
).reset_index(drop=True)

print(f'Shape after dedup: {df_clean.shape}')

Duplicate rows found: 2
Shape after dedup: (272, 8)


### Step 5 — Final verification
Expected final state:
- Shape: ~(272, 8) — 8 cols because we added `perf_was_missing`
- Zero nulls across all columns
- Department has exactly 5 clean values
- Age range: 18–70, Tenure min: 0

In [90]:
print('── Final null check ──')
print(df_clean.isnull().sum())  # all zeros

print(f'\nFinal shape: {df_clean.shape}')

print(f'\nAge range   : {df_clean["age"].min():.1f} - {df_clean["age"].max():.1f}')
print(f'Salary range: {df_clean["salary"].min():,.0f} - {df_clean["salary"].max():,.0f}')
print(f'Tenure min  : {df_clean["tenure_years"].min()}')
print(f'Departments : {sorted(df_clean["department"].unique())}')
print(f'\nperf_was_missing distribution:')
print(df_clean['perf_was_missing'].value_counts())

df_clean.to_csv('data/processed/employee_clean.csv', index=False)
print('\nSaved: employee_clean.csv')

── Final null check ──
employee_id          0
age                  0
salary               0
department           0
tenure_years         0
performance_score    0
left_company         0
perf_was_missing     0
dtype: int64

Final shape: (272, 8)

Age range   : 18.0 - 70.0
Salary range: 17,925 - 450,000
Tenure min  : 0.0
Departments : ['Engineering', 'Finance', 'Hr', 'Marketing', 'Sales']

perf_was_missing distribution:
perf_was_missing
0    251
1     21
Name: count, dtype: int64

Saved: employee_clean.csv
